In [ ]:
!pip install transformers datasets torch torchaudio librosa captum

In [ ]:
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
from datasets import load_dataset
import torch
import librosa

import torch.nn.functional as F
from captum.attr import IntegratedGradients

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import os
from scipy.interpolate import interp1d

### Δείγμα αναφοράς

In [ ]:
TARGET_IDX = 35

### Φόρτωση finetuned μοντέλου

In [ ]:
model_id = "VasilisAsim/data2vec-finetuned-TESS_full"
model_ft = AutoModelForAudioClassification.from_pretrained(model_id)
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)
model_ft.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model_ft.to(device)

### Φόρτωση dataset (speaker-independent)

In [ ]:
from datasets import load_dataset, Audio, DatasetDict

def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))   # για το filter

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")

    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

In [ ]:
dataset = load_tess_speaker_independent()
test_dataset = dataset["test"]

### Wrapper για συμβατότητα με Captum

In [ ]:
def forward_func(input_values):
      """
    Wrapper στο finetuned μοντέλο που επιστρέφει softmax πιθανότητες
    αντί για raw logits, που απαιτείται από το Captum IntegratedGradients, το
    οποίο περιμένει tensor έξοδο.
    """
    outputs = model_ft(input_values)
    return F.softmax(outputs.logits, dim=-1)

In [ ]:
ig = IntegratedGradients(forward_func)

### Βοηθητική συναρτήση για την αποθήκευση attribution

In [ ]:
def resample_attribution(attr, target_frames=50):
    """
    Παίρνει την απόλυτη τιμή ενός attribution vector μεταβλητού μήκους και το
    δειγματοληπτεί σε target_frames σημεία, κανονικοποιώντας ώστε το άθροισμα να
    ισούται με 1. Χρησιμοποιείται ώστε τα διαφορετικά attributions να είναι αποθηκεύσιμα σε
    ενιαία μορφή (.npy, 50 segment)
    """
    attr = np.abs(attr)
    x_original = np.linspace(0, 1, len(attr))
    x_target = np.linspace(0, 1, target_frames)
    f = interp1d(x_original, attr, kind='linear')
    resampled = f(x_target)
    resampled = resampled / (resampled.sum() + 1e-10)
    return resampled

save_dir = "/content/drive/MyDrive/Colab Notebooks/SER_Thesis/01_data2vec_TESS/attributions"
os.makedirs(save_dir, exist_ok=True)

### Εξαγωγή προβλέψεων ανά δείγμα

In [ ]:
def explain_sample(idx, split="test"):
    """
    Υπολογισμός Integrated Gradients attribution για ένα δείγμα. Φόρτωση του ήχου,
    έβρεση προβλεπόμενης κλάσης και υπολογισμός attribution για την κλάση αυτή με μηδενικό
    baseline
    """
    sample = dataset[split][idx]
    speech = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    if sr != feature_extractor.sampling_rate:
        speech = librosa.resample(speech, orig_sr=sr, target_sr=feature_extractor.sampling_rate)

    inputs = feature_extractor(speech, sampling_rate=feature_extractor.sampling_rate, return_tensors="pt")
    input_values = inputs["input_values"].to(device).requires_grad_()

    true_label = sample["emotion"]  # προσάρμοσε το όνομα του πεδίου αν διαφέρει

    with torch.no_grad():
        pred_class = forward_func(input_values).argmax(dim=-1).item()

    baseline = torch.zeros_like(input_values)
    attributions, delta = ig.attribute(
        input_values, baselines=baseline, target=pred_class,
        n_steps=50, return_convergence_delta=True
    )

    return speech, attributions.squeeze().detach().cpu().numpy(), pred_class, true_label, delta.item()

In [ ]:
def find_samples_by_class(target_label, split="test"):
  """
  Περνά όλα τα δείγματα μιας συγκεκριμένης κλάσης από το μοντέλο και τα χωρίζει
  σε σωστά ταξινομημένα και λανθασμένα ταξινομημένα, ώστε να εντοπιστoύν οι λανθασμένες
  προβλέψεις και οι κατηγορίες τους.
  """
  indices = [i for i, label in enumerate(dataset[split]["emotion"]) if label == target_label]
  correct, misclassified = [], []

  for idx in tqdm(indices, desc=f"Επεξεργασία '{target_label}' δειγμάτων"):
      _, _, pred_class, true_label, _ = explain_sample(idx, split=split)
      pred_label = model_ft.config.id2label[pred_class]
      if pred_label == true_label:
          correct.append(idx)
      else:
          misclassified.append((idx, pred_label))

  return correct, misclassified

In [ ]:
angry_correct, angry_wrong = find_samples_by_class("angry")
print(f"Σωστά: {len(angry_correct)}, Λάθος: {len(angry_wrong)}")
print(angry_wrong[:5])

###Plotting : idx=35 (angry > sad)

In [ ]:
speech, attr, pred_class, true_label, delta = explain_sample(TARGET_IDX, split="test")

attr_abs = np.abs(attr)
window = 400
attr_smooth = np.convolve(attr_abs, np.ones(window)/window, mode='same')

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(speech, color='gray')
axes[0].set_title(f"Κυματομορφή — True label: {true_label}")

axes[1].plot(attr_smooth, color='red')
axes[1].set_title(f"Integrated Gradients — Predicted: {model_ft.config.id2label[pred_class]}")
plt.tight_layout()
plt.show()

resampled_ig = resample_attribution(attr, target_frames=50)
np.save(f"{save_dir}/IG_idx{TARGET_IDX}.npy", resampled_ig)
print(f"Αποθηκεύτηκε: IG_idx{TARGET_IDX}.npy")